# Create Final Results CSV

This notebook creates the final results CSV file with all required columns for the hub prioritization analysis.

**Inputs:**
1. Base hub data (e.g., `grouped_hubs_ready_for_scoring_*.csv`) - for Line_Unique and base columns
2. Scored hubs (`scored_hubs_final.csv`) - from COMPLETE_TRANSIT_PIPELINE.ipynb (Monte Carlo scores)
3. Hebrew line names CSV (`hebrew_line_names.csv`) - for Line_Names column (optional)
4. Line status CSV (`line_status.csv`) - for NumLinesStatus_* columns (optional)

**Process:**
1. Load base hub data
2. Create derived columns (x, y, Metro, LocationForChart, TransferRate, etc.)
3. Add Hebrew modes and line names
4. Normalize scoring columns (TotalDemand uses log10 + per-HubType normalization)
5. Load Monte Carlo scores and rankings from scored_hubs_final.csv
6. Add line status counts
7. Export final results

**Output:**
- `data/results/hubs_final_results_YYYYMMDD.csv` with all columns

## Setup

In [ ]:
import pandas as pd
import numpy as np
import ast
import re
from pathlib import Path
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

# Setup paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
RESULTS_DIR = DATA_DIR / "results"

# Create directories if needed
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Results directory: {RESULTS_DIR}")

## Configuration

Configure paths to input files here.

In [ ]:
# ===========================================
# CONFIGURATION - Update paths as needed
# ===========================================

# Base hub data file (with Line_Unique column)
HUBS_FILE = NOTEBOOKS_DIR / 'grouped_hubs_ready_for_scoring_21082025.csv'

# Scored hubs file from COMPLETE_TRANSIT_PIPELINE.ipynb
# Contains: Average_Simulated_Score, Overall_Rank, Rank_within_HubType
SCORED_HUBS_FILE = RESULTS_DIR / 'scored_hubs_final.csv'

# Hebrew line names CSV (LineName -> Line_n_Mode)
HEBREW_LINES_FILE = DATA_DIR / 'hebrew_line_names.csv'

# Line status CSV (LineName -> Status Id)
LINE_STATUS_FILE = DATA_DIR / 'line_status.csv'

# Line name corrections (optional)
LINE_CORRECTIONS_FILE = DATA_DIR / 'line_name_corrections.csv'

# ===========================================
# Hebrew mappings
# ===========================================

# Mode Hebrew names
MODE_HEBREW_MAP = {
    'BRT': 'BRT',
    'Cable': 'רכבל',
    'Funicular': 'פוניקולר',
    'HighSpeed Rail': 'רכבת מהירה',
    'Interurban Rail': 'רכבת בינעירונית',
    'LRT': 'רק"ל',
    'Metro': 'מטרו',
    'Suburban Rail': 'רכבת פרברית'
}

# Hub type Hebrew names
HUBTYPE_HEBREW_MAP = {
    'Train Station': 'ארצי',
    'Metropolitan': 'מטרופוליני',
    'Local': 'עירוני'
}

# Location category Hebrew names
LOCATION_HEBREW_MAP = {
    1: 'חוץ',      # Outer
    2: 'טבעת',     # Ring
    3: 'גלעין'     # Core
}

print("Configuration loaded.")

## Step 1: Load Base Hub Data

In [ ]:
# Load base hub data
if not HUBS_FILE.exists():
    raise FileNotFoundError(f"Hub data file not found: {HUBS_FILE}")

print(f"Loading hub data from: {HUBS_FILE.name}")
hubs_df = pd.read_csv(HUBS_FILE, encoding='utf-8')

print(f"\nLoaded {len(hubs_df)} hubs")

# Check for and remove duplicate groups in base data
if 'group' in hubs_df.columns:
    duplicate_groups = hubs_df['group'].duplicated().sum()
    if duplicate_groups > 0:
        print(f"⚠️ Found {duplicate_groups} duplicate group IDs in base data")
        print(f"  Removing duplicates (keeping first occurrence)...")
        hubs_df = hubs_df.drop_duplicates(subset=['group'], keep='first')
        print(f"  Hubs after deduplication: {len(hubs_df)}")
    else:
        print(f"✓ No duplicate group IDs found")

print(f"Columns ({len(hubs_df.columns)}): {hubs_df.columns.tolist()}")

# Show sample
display(hubs_df.head(3))

## Step 2: Create Coordinate Columns (x, y)

Extract x and y coordinates from the centroid column.

In [ ]:
def extract_coords_from_centroid(centroid_str):
    """
    Extract x, y coordinates from centroid string.
    
    Input format: 'POINT (35.582038147824136 33.21058342039613)'
    Returns: (x, y) tuple
    """
    if pd.isna(centroid_str):
        return None, None
    
    try:
        # Extract numbers from POINT string
        match = re.search(r'POINT\s*\(([\d.-]+)\s+([\d.-]+)\)', str(centroid_str))
        if match:
            x = float(match.group(1))
            y = float(match.group(2))
            return x, y
    except Exception:
        pass
    
    return None, None

# Extract coordinates
print("Extracting x, y coordinates from centroid...")
coords = hubs_df['centroid'].apply(extract_coords_from_centroid)
hubs_df['x'] = coords.apply(lambda c: c[0])
hubs_df['y'] = coords.apply(lambda c: c[1])

print(f"\n✓ Coordinates extracted")
print(f"  Hubs with valid coordinates: {hubs_df['x'].notna().sum()} / {len(hubs_df)}")

# Show sample
display(hubs_df[['group', 'centroid', 'x', 'y']].head())

## Step 3: Create Metro Column

Extract metropolitan area from 'area' column or define based on region.

In [ ]:
# Create Metro column from area
# The 'area' column contains the general region (e.g., 'צפון', 'מרכז', etc.)
# We'll use it as the Metro column

if 'area' in hubs_df.columns:
    hubs_df['Metro'] = hubs_df['area']
    print(f"✓ Metro column created from 'area'")
    print(f"\nMetro distribution:")
    display(hubs_df['Metro'].value_counts())
else:
    print("⚠️ 'area' column not found - Metro column not created")

## Step 4: Create LocationForChart Column (Hebrew)

Map Location_category to Hebrew names for charts.

In [ ]:
# Create LocationForChart column
if 'Location_category' in hubs_df.columns:
    hubs_df['LocationForChart'] = hubs_df['Location_category'].map(LOCATION_HEBREW_MAP)
    print(f"✓ LocationForChart column created")
    print(f"\nLocationForChart distribution:")
    display(hubs_df['LocationForChart'].value_counts())
else:
    print("⚠️ 'Location_category' column not found")

## Step 5: Create TransferRate Column

In [ ]:
# Create TransferRate = TotalTransfers / TotalDemand
if 'TotalTransfers' in hubs_df.columns and 'TotalDemand' in hubs_df.columns:
    hubs_df['TransferRate'] = hubs_df['TotalTransfers'] / hubs_df['TotalDemand'].replace(0, np.nan)
    hubs_df['TransferRate'] = hubs_df['TransferRate'].fillna(0)
    
    print(f"✓ TransferRate column created")
    print(f"\nTransferRate statistics:")
    print(hubs_df['TransferRate'].describe())
else:
    print("⚠️ Required columns not found for TransferRate")

## Step 6: Create Total Population and Employment Columns (2050)

In [ ]:
# Create TotalPop_2050 and TotalEmp_2050
pop_cols = ['pop_0_500', 'pop_500_1000', 'pop_1000_1500']
emp_cols = ['emp_0_500', 'emp_500_1000', 'emp_1000_1500']

# Check which columns exist
existing_pop_cols = [c for c in pop_cols if c in hubs_df.columns]
existing_emp_cols = [c for c in emp_cols if c in hubs_df.columns]

if existing_pop_cols:
    hubs_df['TotalPop_2050'] = hubs_df[existing_pop_cols].sum(axis=1)
    print(f"✓ TotalPop_2050 created from: {existing_pop_cols}")
else:
    print("⚠️ Population columns not found")

if existing_emp_cols:
    hubs_df['TotalEmp_2050'] = hubs_df[existing_emp_cols].sum(axis=1)
    print(f"✓ TotalEmp_2050 created from: {existing_emp_cols}")
else:
    print("⚠️ Employment columns not found")

# Show sample
if 'TotalPop_2050' in hubs_df.columns:
    display(hubs_df[['group', 'TotalPop_2050', 'TotalEmp_2050']].head())

## Step 7: Create Modes_ForPlot Column (Hebrew Mode Names)

In [ ]:
def parse_mode_planned(mode_str):
    """
    Parse Mode_Planned string to list of mode names.
    
    Input: "['Interurban Rail']" or "['BRT', 'LRT']"
    Output: ['Interurban Rail'] or ['BRT', 'LRT']
    """
    if pd.isna(mode_str):
        return []
    
    try:
        # Try to parse as Python literal
        modes = ast.literal_eval(str(mode_str))
        if isinstance(modes, list):
            return modes
        return [modes]
    except:
        # Fallback: remove brackets and split
        mode_str = str(mode_str).strip('[]').replace("'", "")
        return [m.strip() for m in mode_str.split(',') if m.strip()]

def modes_to_hebrew(modes_list):
    """
    Convert list of mode names to Hebrew string.
    """
    hebrew_modes = []
    for mode in modes_list:
        hebrew_name = MODE_HEBREW_MAP.get(mode, mode)
        hebrew_modes.append(hebrew_name)
    return ', '.join(hebrew_modes)

# Create Modes_ForPlot column
if 'Mode_Planned' in hubs_df.columns:
    print("Creating Modes_ForPlot column...")
    hubs_df['modes_list'] = hubs_df['Mode_Planned'].apply(parse_mode_planned)
    hubs_df['Modes_ForPlot'] = hubs_df['modes_list'].apply(modes_to_hebrew)
    hubs_df.drop('modes_list', axis=1, inplace=True)
    
    print(f"✓ Modes_ForPlot column created")
    
    # Show sample
    display(hubs_df[['group', 'Mode_Planned', 'Modes_ForPlot']].head(10))
else:
    print("⚠️ 'Mode_Planned' column not found")

## Step 8: Create HubTypeHE Column (Hebrew Hub Type)

In [ ]:
# Create HubTypeHE column
if 'HubType' in hubs_df.columns:
    hubs_df['HubTypeHE'] = hubs_df['HubType'].map(HUBTYPE_HEBREW_MAP)
    # Fill unmapped values with original
    hubs_df['HubTypeHE'] = hubs_df['HubTypeHE'].fillna(hubs_df['HubType'])
    
    print(f"✓ HubTypeHE column created")
    print(f"\nHubTypeHE distribution:")
    display(hubs_df['HubTypeHE'].value_counts())
else:
    print("⚠️ 'HubType' column not found")

## Step 9: Create HubType_Filtered Column

Filter hubs in TLV Metro with low TransferRate (<80%) and low line count (≤2).

In [ ]:
# Create HubType_Filtered column
# A hub is filtered if: Metro="TLV" (or מרכז) AND TransferRate < 0.8 AND Total_Unique_Lines <= 2

def is_hub_filtered(row):
    """
    Determine if a hub should be filtered.
    Returns True if hub is filtered (should potentially be excluded).
    """
    metro = str(row.get('Metro', '')).lower()
    transfer_rate = row.get('TransferRate', 1)
    total_lines = row.get('Total_Unique_Lines', 10)
    
    # Check if in TLV/Center metro area
    is_tlv = 'tlv' in metro or 'מרכז' in metro or 'center' in metro.lower()
    
    # Check filter conditions
    if is_tlv and transfer_rate < 0.8 and total_lines <= 2:
        return True
    return False

if 'Metro' in hubs_df.columns and 'TransferRate' in hubs_df.columns:
    hubs_df['HubType_Filtered'] = hubs_df.apply(is_hub_filtered, axis=1)
    
    filtered_count = hubs_df['HubType_Filtered'].sum()
    print(f"✓ HubType_Filtered column created")
    print(f"  Filtered hubs: {filtered_count} / {len(hubs_df)}")
else:
    print("⚠️ Required columns not found for HubType_Filtered")

## Step 10: Create BusTERMINAL_Clone Column

In [ ]:
# Create BusTERMINAL_Clone column (clone of bus_terminal)
if 'bus_terminal' in hubs_df.columns:
    hubs_df['BusTERMINAL_Clone'] = hubs_df['bus_terminal']
    print(f"✓ BusTERMINAL_Clone column created")
else:
    print("⚠️ 'bus_terminal' column not found")

## Step 11: Create TotalNumLines Column

In [ ]:
# Create TotalNumLines - count of all lines in the group
line_count_cols = [
    'BRT Lines', 'Cable Line Lines', 'Funicular Lines', 
    'HighSpeed Rail Lines', 'Interurban Rail Lines', 
    'LRT Lines', 'Metro Lines', 'Suburban Rail Lines'
]

existing_line_cols = [c for c in line_count_cols if c in hubs_df.columns]

if existing_line_cols:
    hubs_df['TotalNumLines'] = hubs_df[existing_line_cols].sum(axis=1)
    print(f"✓ TotalNumLines column created from: {len(existing_line_cols)} columns")
    print(f"\nTotalNumLines statistics:")
    print(hubs_df['TotalNumLines'].describe())
elif 'Total_Unique_Lines' in hubs_df.columns:
    hubs_df['TotalNumLines'] = hubs_df['Total_Unique_Lines']
    print(f"✓ TotalNumLines column created from Total_Unique_Lines")
else:
    print("⚠️ Line count columns not found")

## Step 12: Create Normalized Scoring Columns

Normalize scoring columns to 1-10 range.

In [ ]:
def normalize_to_1_10(series):
    """
    Normalize a series to 1-10 range.
    """
    min_val = series.min()
    max_val = series.max()
    
    if max_val == min_val:
        return pd.Series([5.5] * len(series), index=series.index)
    
    return 1 + 9 * (series - min_val) / (max_val - min_val)

def normalize_by_hub_type(df, col, new_col):
    """
    Normalize a column to 1-10 range within each hub type.
    This matches the normalization method in COMPLETE_TRANSIT_PIPELINE.ipynb
    """
    df[new_col] = np.nan
    
    for hub_type in df['HubType'].unique():
        mask = df['HubType'] == hub_type
        values = df.loc[mask, col]
        values_nonzero = values[values > 0]
        
        if len(values_nonzero) > 0:
            min_val, max_val = values_nonzero.min(), values_nonzero.max()
            if max_val > min_val:
                df.loc[mask, new_col] = 1 + (values - min_val) * 9 / (max_val - min_val)
            else:
                df.loc[mask, new_col] = 5.5
        else:
            df.loc[mask, new_col] = 1.0
    
    return df

# Columns to normalize (simple min-max)
normalize_map = {
    'RegionLocation': 'RegionLocation_Norm',
    'score': 'score_Norm',
    'bus_terminal': 'bus_terminal_Norm',
}

print("Normalizing scoring columns to 1-10 range...")

for orig_col, norm_col in normalize_map.items():
    if orig_col in hubs_df.columns:
        hubs_df[norm_col] = normalize_to_1_10(hubs_df[orig_col].fillna(0))
        print(f"  ✓ {norm_col} created")
    else:
        print(f"  ⚠️ {orig_col} not found, skipping {norm_col}")

# Normalize TotalDemand with LOG10 transformation (matching COMPLETE_TRANSIT_PIPELINE.ipynb)
# This prevents extreme skew from mega-stations
if 'TotalDemand' in hubs_df.columns and 'HubType' in hubs_df.columns:
    print(f"\n  Normalizing TotalDemand with log10 transformation...")
    
    # Apply log10 transformation (0 if value is 0)
    hubs_df['LogDemand'] = hubs_df['TotalDemand'].apply(lambda x: 0 if x == 0 else np.log10(x))
    
    # Normalize to 1-10 within each hub type
    hubs_df = normalize_by_hub_type(hubs_df, 'LogDemand', 'TotalDemand_Norm')
    
    print(f"  ✓ TotalDemand_Norm created (log10 + per-HubType normalization)")
    print(f"    TotalDemand_Norm range: {hubs_df['TotalDemand_Norm'].min():.2f} - {hubs_df['TotalDemand_Norm'].max():.2f}")
elif 'TotalDemand' in hubs_df.columns:
    # Fallback: simple normalization without hub type
    hubs_df['LogDemand'] = hubs_df['TotalDemand'].apply(lambda x: 0 if x == 0 else np.log10(x))
    hubs_df['TotalDemand_Norm'] = normalize_to_1_10(hubs_df['LogDemand'])
    print(f"  ✓ TotalDemand_Norm created (log10, no hub type)")
else:
    print(f"  ⚠️ TotalDemand not found, skipping TotalDemand_Norm")

# Create PopEmp_Score_Norm (combined population and employment score)
if 'TotalPop_2050' in hubs_df.columns and 'TotalEmp_2050' in hubs_df.columns:
    # Weighted combination: based on hub type
    # National/Metropolitan: 80% jobs, 20% pop
    # Local: 20% jobs, 80% pop
    
    def calc_pop_emp_score(row):
        hub_type = str(row.get('HubType', '')).lower()
        pop = row.get('TotalPop_2050', 0)
        emp = row.get('TotalEmp_2050', 0)
        
        if 'local' in hub_type or 'עירוני' in hub_type:
            return 0.8 * pop + 0.2 * emp
        else:
            return 0.2 * pop + 0.8 * emp
    
    hubs_df['PopEmp_Score'] = hubs_df.apply(calc_pop_emp_score, axis=1)
    hubs_df['PopEmp_Score_Norm'] = normalize_to_1_10(hubs_df['PopEmp_Score'])
    print(f"  ✓ PopEmp_Score_Norm created")

print("\nNormalized columns created.")

## Step 13: Load Monte Carlo Scores from scored_hubs_final.csv

Load the pre-calculated Monte Carlo scores from COMPLETE_TRANSIT_PIPELINE.ipynb output.

**Columns loaded:**
- `Average_Simulated_Score` - MC aggregated score (renamed to TotalScore_MC)
- `Overall_Rank` - Overall ranking (renamed to Rank_TS_MC)
- `Rank_within_HubType` - Ranking within hub type

In [ ]:
# Load Monte Carlo scores from COMPLETE_TRANSIT_PIPELINE.ipynb output
# Instead of recalculating, we use the pre-computed scores

print("Loading Monte Carlo scores from scored_hubs_final.csv...")
print(f"  Hubs before merge: {len(hubs_df)}")

# Check if scored hubs file exists
if not SCORED_HUBS_FILE.exists():
    # Try alternative locations
    alternative_paths = [
        DATA_DIR / 'scored_hubs_final.csv',
        PROJECT_ROOT / 'scored_hubs_final.csv',
        NOTEBOOKS_DIR / 'scored_hubs_final.csv',
    ]
    
    for path in alternative_paths:
        if path.exists():
            SCORED_HUBS_FILE_FOUND = path
            break
    else:
        SCORED_HUBS_FILE_FOUND = None
else:
    SCORED_HUBS_FILE_FOUND = SCORED_HUBS_FILE

if SCORED_HUBS_FILE_FOUND is not None:
    print(f"  Loading from: {SCORED_HUBS_FILE_FOUND}")
    scored_df = pd.read_csv(SCORED_HUBS_FILE_FOUND, encoding='utf-8')
    
    print(f"  Loaded {len(scored_df)} scored hubs")
    
    # Check for required columns
    score_columns_map = {
        'Average_Simulated_Score': 'TotalScore_MC',
        'Overall_Rank': 'Rank_TS_MC',
        'Rank_within_HubType': 'Rank_By_TS_MC_By_Metro'
    }
    
    # Merge scores into main dataframe
    merge_cols = ['group']
    for orig_col, new_col in score_columns_map.items():
        if orig_col in scored_df.columns:
            merge_cols.append(orig_col)
    
    if len(merge_cols) > 1:
        # Rename columns before merge
        scored_subset = scored_df[merge_cols].copy()
        scored_subset = scored_subset.rename(columns=score_columns_map)
        
        # IMPORTANT: Drop duplicates to prevent row multiplication during merge
        duplicates_before = len(scored_subset)
        scored_subset = scored_subset.drop_duplicates(subset=['group'], keep='first')
        duplicates_removed = duplicates_before - len(scored_subset)
        if duplicates_removed > 0:
            print(f"  ⚠️ Removed {duplicates_removed} duplicate groups from scored_hubs_final.csv")
        
        # Merge using left join to preserve original row count
        hubs_df = hubs_df.merge(scored_subset, on='group', how='left')
        
        print(f"  Hubs after merge: {len(hubs_df)}")
        
        # Verify no duplicates were created
        if len(hubs_df) > duplicates_before:
            print(f"  ⚠️ WARNING: Row count increased after merge!")
        
        print(f"\n✓ Monte Carlo scores merged successfully")
        
        # Show statistics
        if 'TotalScore_MC' in hubs_df.columns:
            print(f"  TotalScore_MC range: {hubs_df['TotalScore_MC'].min():.3f} - {hubs_df['TotalScore_MC'].max():.3f}")
            print(f"  TotalScore_MC mean: {hubs_df['TotalScore_MC'].mean():.3f}")
            print(f"  Hubs with scores: {hubs_df['TotalScore_MC'].notna().sum()} / {len(hubs_df)}")
        
        if 'Rank_TS_MC' in hubs_df.columns:
            print(f"  Rank_TS_MC range: {int(hubs_df['Rank_TS_MC'].min())} - {int(hubs_df['Rank_TS_MC'].max())}")
    else:
        print(f"  ⚠️ Required score columns not found in {SCORED_HUBS_FILE_FOUND.name}")
        print(f"  Available columns: {scored_df.columns.tolist()}")

else:
    print(f"⚠️ Scored hubs file not found!")
    print(f"  Expected: {SCORED_HUBS_FILE}")
    print(f"  Please run COMPLETE_TRANSIT_PIPELINE.ipynb first to generate scored_hubs_final.csv")
    print(f"\n  Score columns will not be added.")

## Step 14: Verify Rankings

Verify the loaded rankings and show top hubs.

In [ ]:
# Verify rankings loaded from scored_hubs_final.csv

print("Verifying loaded rankings...")

# Check if rank columns exist (loaded from scored_hubs_final.csv)
if 'Rank_TS_MC' in hubs_df.columns:
    print(f"✓ Rank_TS_MC loaded (overall rank)")
    print(f"  Range: {int(hubs_df['Rank_TS_MC'].min())} - {int(hubs_df['Rank_TS_MC'].max())}")
else:
    print("⚠️ Rank_TS_MC not found - scores may not have been loaded")

if 'Rank_By_TS_MC_By_Metro' in hubs_df.columns:
    print(f"✓ Rank_By_TS_MC_By_Metro loaded (rank within hub type)")
else:
    print("⚠️ Rank_By_TS_MC_By_Metro not found")

# Show top 10 hubs
if 'Rank_TS_MC' in hubs_df.columns:
    print(f"\nTop 10 hubs by TotalScore_MC:")
    top_cols = ['group', 'TotalScore_MC', 'Rank_TS_MC']
    if 'address' in hubs_df.columns:
        top_cols.insert(1, 'address')
    if 'HubType' in hubs_df.columns:
        top_cols.insert(2, 'HubType')
    if 'Rank_By_TS_MC_By_Metro' in hubs_df.columns:
        top_cols.append('Rank_By_TS_MC_By_Metro')
    
    # Filter to only existing columns
    top_cols = [c for c in top_cols if c in hubs_df.columns]
    
    display(hubs_df.nsmallest(10, 'Rank_TS_MC')[top_cols])

## Step 15: Add Hebrew Line Names

Parse Line_Unique and join with Hebrew line names CSV to create Line_Names column.

In [ ]:
def parse_line_unique_full(value):
    """
    Parse complete Line_Unique value.
    
    Input: ["['rail_1_1' 'rail_1_2']", "['rail_5_1' 'rail_5_2']"]
    Output: ['rail_1_1', 'rail_1_2', 'rail_5_1', 'rail_5_2']
    """
    if pd.isna(value) or value == '' or value == '[]':
        return []
    
    # Parse outer list structure
    try:
        outer_list = ast.literal_eval(str(value))
        if not isinstance(outer_list, list):
            outer_list = [outer_list]
    except (ValueError, SyntaxError):
        outer_list = [value]
    
    # Parse each element and flatten
    all_lines = []
    for element in outer_list:
        element_str = str(element).strip('[]')
        element_str = element_str.replace("'", "").replace('"', '')
        element_str = element_str.replace('\\r', '').replace('\r', '')
        lines = re.split(r'[,\s]+', element_str)
        lines = [x.strip() for x in lines if x.strip()]
        all_lines.extend(lines)
    
    # Remove duplicates while preserving order
    seen = set()
    unique_lines = []
    for line in all_lines:
        if line not in seen:
            seen.add(line)
            unique_lines.append(line)
    
    return unique_lines

# Check if Hebrew line names file exists
if HEBREW_LINES_FILE.exists():
    print(f"Loading Hebrew line names from: {HEBREW_LINES_FILE.name}")
    hebrew_names_df = pd.read_csv(HEBREW_LINES_FILE, encoding='utf-8')
    
    # Create mapping dictionary
    if 'LineName' in hebrew_names_df.columns and 'Line_n_Mode' in hebrew_names_df.columns:
        hebrew_map = dict(zip(
            hebrew_names_df['LineName'].astype(str).str.strip(),
            hebrew_names_df['Line_n_Mode'].astype(str).str.strip()
        ))
        
        print(f"  Loaded {len(hebrew_map)} Hebrew line mappings")
        
        # Parse Line_Unique and map to Hebrew names
        def get_hebrew_line_names(line_unique_value):
            line_ids = parse_line_unique_full(line_unique_value)
            hebrew_names = [hebrew_map.get(lid, lid) for lid in line_ids]
            return hebrew_names
        
        if 'Line_Unique' in hubs_df.columns:
            hubs_df['Line_Names'] = hubs_df['Line_Unique'].apply(get_hebrew_line_names)
            # Convert list to string for CSV
            hubs_df['Line_Names'] = hubs_df['Line_Names'].apply(str)
            print(f"  ✓ Line_Names column created")
        else:
            print(f"  ⚠️ Line_Unique column not found")
    else:
        print(f"  ⚠️ Required columns not found in Hebrew names file")
else:
    print(f"⚠️ Hebrew line names file not found: {HEBREW_LINES_FILE}")
    print("  Line_Names column will not be created")
    print("  To add Hebrew line names, create a CSV with columns: LineName, Line_n_Mode")

## Step 16: Add Line Status Columns

Parse Line_Unique and join with line status CSV to create NumLinesStatus_* columns.

In [ ]:
# Check if line status file exists
if LINE_STATUS_FILE.exists():
    print(f"Loading line status data from: {LINE_STATUS_FILE.name}")
    line_status_df = pd.read_csv(LINE_STATUS_FILE, encoding='utf-8')
    
    if 'LineName' in line_status_df.columns and 'Status Id' in line_status_df.columns:
        # Clean line names
        line_status_df['LineName'] = line_status_df['LineName'].astype(str).str.strip()
        line_status_df['LineName'] = line_status_df['LineName'].str.replace(r'[\r\n]', '', regex=True)
        
        # Create status mapping
        status_map = dict(zip(line_status_df['LineName'], line_status_df['Status Id']))
        
        print(f"  Loaded {len(status_map)} line status mappings")
        print(f"  Unique status IDs: {sorted(line_status_df['Status Id'].dropna().unique())}")
        
        # Load corrections if available
        corrections_map = {}
        if LINE_CORRECTIONS_FILE.exists():
            corrections_df = pd.read_csv(LINE_CORRECTIONS_FILE, encoding='utf-8')
            if 'LineName' in corrections_df.columns and 'LineName_Correct' in corrections_df.columns:
                corrections_map = dict(zip(
                    corrections_df['LineName'].astype(str).str.strip(),
                    corrections_df['LineName_Correct'].astype(str).str.strip()
                ))
                print(f"  Loaded {len(corrections_map)} line name corrections")
        
        def get_status_counts(line_unique_value):
            """
            Get count of lines by status ID for a hub.
            Returns dict: {status_id: count}
            """
            line_ids = parse_line_unique_full(line_unique_value)
            
            # Apply corrections
            corrected_ids = [corrections_map.get(lid, lid) for lid in line_ids]
            
            # Get status for each line
            status_counts = {}
            for lid in corrected_ids:
                status = status_map.get(lid)
                if pd.notna(status):
                    status_key = int(status)
                    status_counts[status_key] = status_counts.get(status_key, 0) + 1
            
            return status_counts
        
        if 'Line_Unique' in hubs_df.columns:
            # Get status counts for each hub
            status_dicts = hubs_df['Line_Unique'].apply(get_status_counts)
            
            # Create columns for each status (0-7)
            for status_id in range(8):
                col_name = f'NumLinesStatus_{status_id}'
                hubs_df[col_name] = status_dicts.apply(lambda d: d.get(status_id, 0))
            
            # Create total column
            status_cols = [f'NumLinesStatus_{i}' for i in range(8)]
            hubs_df['TotalLinesAllStatuses'] = hubs_df[status_cols].sum(axis=1)
            
            print(f"  ✓ Status columns created: {status_cols + ['TotalLinesAllStatuses']}")
        else:
            print(f"  ⚠️ Line_Unique column not found")
    else:
        print(f"  ⚠️ Required columns not found in line status file")
else:
    print(f"⚠️ Line status file not found: {LINE_STATUS_FILE}")
    print("  Status columns will not be created")
    print("  To add line status, create a CSV with columns: LineName, Status Id")

## Step 17: Add HubName Column (if available)

In [ ]:
# Check if hub names file exists
hub_names_file = DATA_DIR / 'hub_names.csv'

print(f"Checking for hub names file...")
print(f"  Hubs before merge: {len(hubs_df)}")

if hub_names_file.exists():
    print(f"Loading hub names from: {hub_names_file.name}")
    hub_names_df = pd.read_csv(hub_names_file, encoding='utf-8')
    
    if 'group' in hub_names_df.columns and 'HubName' in hub_names_df.columns:
        # IMPORTANT: Drop duplicates to prevent row multiplication during merge
        hub_names_subset = hub_names_df[['group', 'HubName']].copy()
        duplicates_before = len(hub_names_subset)
        hub_names_subset = hub_names_subset.drop_duplicates(subset=['group'], keep='first')
        duplicates_removed = duplicates_before - len(hub_names_subset)
        if duplicates_removed > 0:
            print(f"  ⚠️ Removed {duplicates_removed} duplicate groups from hub_names.csv")
        
        # Merge hub names
        hubs_df = hubs_df.merge(
            hub_names_subset,
            on='group',
            how='left'
        )
        
        print(f"  Hubs after merge: {len(hubs_df)}")
        print(f"  ✓ HubName column added")
        print(f"  Hubs with names: {hubs_df['HubName'].notna().sum()} / {len(hubs_df)}")
    else:
        print(f"  ⚠️ Required columns not found in hub names file")
else:
    print(f"HubName file not found: {hub_names_file}")
    
    # Create HubName from address if not available
    if 'HubName' not in hubs_df.columns and 'address' in hubs_df.columns:
        # Extract first line of address as hub name
        hubs_df['HubName'] = hubs_df['address'].apply(
            lambda x: str(x).split('\n')[0].strip() if pd.notna(x) else ''
        )
        print(f"  ✓ HubName column created from address")

## Step 18: Reorder Columns

Arrange columns in a logical order for the final output.

In [ ]:
# Define desired column order
column_order = [
    # Identifiers
    'group', 'geometry', 'x', 'y', 'HubName', 'h3_index', 'node',
    
    # Modes and Lines
    'Mode_Planned', 'Modes_ForPlot', 'Model', 'Line_Unique', 'Line_Names',
    
    # Location
    'address', 'area', 'Metro', 'location', 'LocationForChart',
    
    # Demand
    'TotalDemand', 'TotalTransfers', 'TransferRate', 'Total_Unique_Lines',
    
    # Line counts by mode
    'BRT Lines', 'Cable Line Lines', 'Funicular Lines', 'HighSpeed Rail Lines',
    'Interurban Rail Lines', 'LRT Lines', 'Metro Lines', 'Suburban Rail Lines',
    
    # Geometry
    'centroid',
    
    # Population and Employment
    'pop_0_500', 'emp_0_500', 'pop_500_1000', 'emp_500_1000',
    'pop_1000_1500', 'emp_1000_1500', 'TotalPop_2050', 'TotalEmp_2050',
    
    # Bus terminal
    'id', 'term_type',
    
    # Scoring inputs
    'Region_category', 'Location_category', 'RegionLocation', 'Num_Modes',
    'score', 'bus_terminal',
    
    # Hub classification
    'HubType', 'HubType_Filtered', 'HubTypeHE', 'BusTERMINAL_Clone',
    
    # Normalized scores
    'RegionLocation_Norm', 'score_Norm', 'bus_terminal_Norm',
    'TotalDemand_Norm', 'PopEmp_Score_Norm',
    
    # Final scores and rankings
    'TotalScore_MC', 'Rank_TS_MC', 'Rank_By_TS_MC_By_Metro', 'TotalNumLines',
    
    # Line status columns
    'NumLinesStatus_0', 'NumLinesStatus_1', 'NumLinesStatus_2', 'NumLinesStatus_3',
    'NumLinesStatus_4', 'NumLinesStatus_5', 'NumLinesStatus_6', 'NumLinesStatus_7',
    'TotalLinesAllStatuses'
]

# Get columns that exist in the dataframe
existing_columns = [c for c in column_order if c in hubs_df.columns]

# Add any columns not in the order list at the end
other_columns = [c for c in hubs_df.columns if c not in column_order]

# Reorder dataframe
final_columns = existing_columns + other_columns
hubs_df = hubs_df[final_columns]

print(f"Columns reordered. Total: {len(hubs_df.columns)}")
print(f"\nFinal column order:")
for i, col in enumerate(hubs_df.columns, 1):
    print(f"  {i:2d}. {col}")

## Step 19: Final Summary

In [ ]:
print("="*80)
print("FINAL RESULTS SUMMARY")
print("="*80)

print(f"\nTotal hubs: {len(hubs_df)}")
print(f"Total columns: {len(hubs_df.columns)}")

# Check for duplicates
if 'group' in hubs_df.columns:
    duplicate_count = hubs_df['group'].duplicated().sum()
    if duplicate_count > 0:
        print(f"\n⚠️ WARNING: Found {duplicate_count} duplicate rows!")
        print(f"  Removing duplicates before export...")
        hubs_df = hubs_df.drop_duplicates(subset=['group'], keep='first')
        print(f"  Hubs after deduplication: {len(hubs_df)}")
    else:
        print(f"\n✓ No duplicate rows found")

# Show hub type distribution
if 'HubType' in hubs_df.columns:
    print(f"\nHub Type Distribution:")
    display(hubs_df['HubType'].value_counts())

# Show metro distribution
if 'Metro' in hubs_df.columns:
    print(f"\nMetro Distribution:")
    display(hubs_df['Metro'].value_counts())

# Score statistics
if 'TotalScore_MC' in hubs_df.columns:
    print(f"\nScore Statistics:")
    print(hubs_df['TotalScore_MC'].describe())

# Show sample
print(f"\nSample of final data:")
display(hubs_df.head(5))

## Step 20: Export Final Results

In [ ]:
# Create output filename with timestamp
timestamp = pd.Timestamp.now().strftime('%Y%m%d')
output_file = RESULTS_DIR / f'hubs_final_results_{timestamp}.csv'

# Sort by rank before exporting
if 'Rank_TS_MC' in hubs_df.columns:
    hubs_df = hubs_df.sort_values('Rank_TS_MC')

# Export to CSV with UTF-8 BOM for Excel compatibility
hubs_df.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"✓ Final results exported to: {output_file.name}")
print(f"\nFull path: {output_file}")
print(f"\nExport Summary:")
print(f"  Total hubs: {len(hubs_df)}")
print(f"  Total columns: {len(hubs_df.columns)}")
print(f"  File size: {output_file.stat().st_size / 1024:.1f} KB")

## Summary

### Columns Created

**Coordinate Columns:**
- `x`, `y` - Extracted from centroid

**Location Columns:**
- `Metro` - Metropolitan area (from area)
- `LocationForChart` - Hebrew location category (גלעין, טבעת, חוץ)

**Demand Columns:**
- `TransferRate` - TotalTransfers / TotalDemand
- `TotalPop_2050` - Sum of population in catchment rings
- `TotalEmp_2050` - Sum of employment in catchment rings

**Hebrew Columns:**
- `Modes_ForPlot` - Hebrew mode names
- `HubTypeHE` - Hebrew hub type
- `Line_Names` - Hebrew line names (if hebrew_line_names.csv provided)

**Classification Columns:**
- `HubType_Filtered` - Filter flag for TLV hubs with low transfer rate
- `BusTERMINAL_Clone` - Clone of bus_terminal
- `TotalNumLines` - Total line count

**Normalized Score Columns:**
- `RegionLocation_Norm` (1-10)
- `score_Norm` (1-10)
- `bus_terminal_Norm` (1-10)
- `TotalDemand_Norm` (1-10, **log10 transformed + per-HubType normalization**)
- `PopEmp_Score_Norm` (1-10)

**Monte Carlo Scoring (loaded from scored_hubs_final.csv):**
- `TotalScore_MC` - Aggregated MC score (from Average_Simulated_Score)
- `Rank_TS_MC` - Overall rank (from Overall_Rank)
- `Rank_By_TS_MC_By_Metro` - Rank within hub type (from Rank_within_HubType)

**Line Status Columns (if line_status.csv provided):**
- `NumLinesStatus_0` through `NumLinesStatus_7`
- `TotalLinesAllStatuses`

### Required Input Files

1. **Base hub data** (required): `grouped_hubs_ready_for_scoring_*.csv`
   - Contains Line_Unique column and base hub data
2. **Scored hubs** (required for scores): `data/results/scored_hubs_final.csv`
   - Output from COMPLETE_TRANSIT_PIPELINE.ipynb
   - Contains: Average_Simulated_Score, Overall_Rank, Rank_within_HubType
3. **Hebrew line names** (optional): `data/hebrew_line_names.csv`
   - Columns: `LineName`, `Line_n_Mode`
4. **Line status** (optional): `data/line_status.csv`
   - Columns: `LineName`, `Status Id`
5. **Line corrections** (optional): `data/line_name_corrections.csv`
   - Columns: `LineName`, `LineName_Correct`
6. **Hub names** (optional): `data/hub_names.csv`
   - Columns: `group`, `HubName`

### TotalDemand Normalization

The `TotalDemand_Norm` column uses the same normalization as COMPLETE_TRANSIT_PIPELINE.ipynb:
1. **Log10 transformation**: Prevents extreme skew from mega-stations
2. **Per-HubType normalization**: Normalized to 1-10 within each hub type category

### Output File

- Location: `data/results/hubs_final_results_YYYYMMDD.csv`
- Encoding: UTF-8 with BOM (Excel compatible)
- Sorted by: Rank_TS_MC (best ranked first)